In [2]:
import pandas as pd
import re
from tqdm import tqdm
import pickle

import nltk
from nltk import pos_tag
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger') 

from sentence_transformers import SentenceTransformer


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\danie\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\danie\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\danie\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
c:\Users\danie\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [45]:
df = pd.read_csv("movie_review.csv")[["id", "sentiment", "review"]]
df

,id,sentiment,review
0,5814_8,1,With all this stuff going down at the moment w...
1,2381_9,1,"\""The Classic War of the Worlds\"" by Timothy H..."
2,7759_3,0,The film starts with a manager (Nicholas Bell)...
3,3630_4,0,It must be assumed that those who praised this...
4,9495_8,1,Superbly trashy and wondrously unpretentious 8...
...,...,...,...
4995,3720_2,0,Is this film a joke? Is it a comedy? Surely it...
4996,4229_10,1,Don't waste time reading my review. Go out and...
4997,8042_3,0,"The secret is...this movie blows. Sorry, but i..."
4998,9669_9,1,After reading the original play I thought it w...


In [46]:
# Defining utilities that do not have to be defined all the time
le = WordNetLemmatizer()

# Define a function to get correct position input for the Lemmatizer.
def get_pos_code(x):
    if x.startswith('J'):
        return 'a'
    elif x.startswith('V'):
        return 'v'
    elif x.startswith('N'):
        return 'n'
    elif x.startswith('R'):
        return 'r'
    else:
        return 'n'

def preprocess_document(doc: str):
    """
    This function preprocesses a textual document into a bag of words.
    Applied steps:
    - Lowercasing
    - Removing HTML tags
    - Removing stopwords
    - Removing punctuation
    - Lemmatization

    """
    # Converting to lowercase
    doc = doc.lower()

    # Removing HTML tags
    doc = re.sub("</?[^>]+>", "", doc)

    # Removing punctuation, but ' (single quotes) are not removed (as stopwords may contain them.)
    punct = str.maketrans("", "", '!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~')
    doc = doc.translate(punct)

    # Removing stopwords
    doc = doc.split(" ")
    doc = [word for word in doc if word not in stopwords.words("English")]

    # Removing the single quote
    doc = " ".join(doc)
    punct = str.maketrans("", "", "'")
    doc = doc.translate(punct)
    doc = word_tokenize(doc)

    # Lemmatization
    doc = pos_tag(doc) # Pos tagging to improve lemmatization process.
    doc = [le.lemmatize(word, get_pos_code(tag)) for word, tag in doc]

    # Joining again into single string for final output
    doc = " ".join(doc)

    return doc
    
    


In [ ]:
df["review"] = tqdm(df["review"].apply(preprocess_document))
# This takes a long time. 

100%|██████████| 5000/5000 [00:00<?, ?it/s]


In [ ]:
# Saving the preprocessed reviews to use them later (without havign to do the whole preprocessing again)
df.to_csv("movie_review_preprocessed.csv")

In [59]:
# Here load the preprocessed DataFrame (so you don't have to perform it each time)
df = pd.read_csv("movie_review_preprocessed.csv", index_col = 0)
df

,id,sentiment,review
0,5814_8,1,stuff go moment mj ive start listen music watc...
1,2381_9,1,classic war world timothy hines entertain film...
2,7759_3,0,film start manager nicholas bell give welcome ...
3,3630_4,0,must assume praised film great film opera ever...
4,9495_8,1,superbly trashy wondrously unpretentious 80 ex...
...,...,...,...
4995,3720_2,0,film joke comedy surely serious thriller sugge...
4996,4229_10,1,waste time read review go see astonishingly go...
4997,8042_3,0,secret isthis movie blow sorry spoilerin bad r...
4998,9669_9,1,read original play think would much difficult ...


In [ ]:
# Using sentence-transformers/all-MiniLM-L6-v2
model_id = "sentence-transformers/all-MiniLM-L6-v2"

# HuggingFace token is secretly stored
with open("token.txt", "r") as token_file:
    hf_token = token_file.readline()

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", token = hf_token)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2452.93it/s]


In [73]:
embeddings = model.encode(list(df["review"]))
print(f"Embedded into {len(embeddings[0])}-dimensional space")

Embedded into 384-dimensional space


In [74]:
# Pickling the embeddings (to quickly open them in new sessions)
pickle.dump(embeddings, open('embeddings.pkl', 'wb'))

In [79]:
# Unpicklings the embeddings (loading them from a file)
loaded = pickle.load(open('embeddings.pkl', 'rb'))
print(f"Loaded embeddings \n{len(loaded)} vectors, {len(loaded[0])} dimensions")

Loaded embeddings 
5000 vectors, 384 dimensions
